# Detection

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import matplotlib.pyplot as plt
from math import ceil, floor
from datetime import datetime 
torch.manual_seed(123)
import torchvision
from torchvision.ops import box_convert, complete_box_iou_loss, box_iou
import math
from torch.utils.data import TensorDataset, DataLoader
import copy

# Normalization 

In [ ]:
def normalize(train, val, test):
    train_tensors = train.tensors[0]  # Get the underlying tensor
    val_tensors = val.tensors[0]  
    test_tensors = test.tensors[0]  

    mean = train_tensors.mean()    # Mean per feature
    std  = train_tensors.std()     # Std per feature
    
    train_norm = (train_tensors - mean) / std 
    val_norm = (val_tensors - mean) / std 
    test_norm = (test_tensors - mean) / std 
    return TensorDataset(train_norm, train.tensors[1]), TensorDataset(val_norm, val.tensors[1]), TensorDataset(test_norm, test.tensors[1])


### Load data and preprocessing

In [ ]:
def convert_to_grid(list_y_true, Hout, Wout):
    Ntot = len(list_y_true)  # Antall bilder
    
    # Tom tensor for alle bilder
    y_true = torch.zeros(Ntot, Hout, Wout, 6)
    
    for img_idx, objects in enumerate(list_y_true):
        for obj in objects:
            pc, x, y, w, h, c = obj
            
            # Finn hvilken celle objektet tilhører
            col = int(x * Wout)  # Hvilken kolonne (0 til Wout-1)
            row = int(y * Hout)  # Hvilken rad (0 til Hout-1)
            
            # Klem til gyldig indeks
            col = min(col, Wout - 1)
            row = min(row, Hout - 1)
            
            # Konverter til lokale koordinater
            x_local = x * Wout - col  # x relativt til cellen
            y_local = y * Hout - row  # y relativt til cellen
            w_local = w * Wout        # w relativt til cellestørrelse
            h_local = h * Hout        # h relativt til cellestørrelse
            
            # Plasser i riktig celle
            y_true[img_idx, row, col] = torch.tensor([pc, x_local, y_local, w_local, h_local, c])
    
    return y_true

In [ ]:
def get_converted_data(
    grid_dimensions: tuple[int, int]
) -> tuple[TensorDataset, TensorDataset, TensorDataset]:
    """
    Get training, validation, and test datasets with labels
    converted into to a given grid size.

    Parameters:
    grid_dimensions: dimensions of the grid

    return: converted train, val, test datasets
    """

    

In [ ]:
def load_data(H_out, W_out):
    list_train = torch.load('data/list_y_true_train.pt')
    list_val = torch.load('data/list_y_true_val.pt')
    list_test = torch.load("data/list_y_true_test.pt")
    label_sets = [list_train, list_val, list_test]

    sample = list_train[0]
    print(type(sample))
    print(sample)
    print(sample.shape if hasattr(sample, 'shape') else len(sample))

    # Also check a single object inside
    if isinstance(sample, (list, tuple)):
        print("First object:", sample[0])
        print("First object shape:", sample[0].shape if hasattr(sample[0], 'shape') else sample[0])

    imgs_train = torch.load("data/detection_train.pt")
    imgs_val = torch.load("data/detection_val.pt")
    imgs_test = torch.load("data/detection_test.pt")
    img_sets = [imgs_train, imgs_val, imgs_test]

    output_datasets = []
    for img_set, label_set in zip(img_sets, label_sets):
        labels_tensor = convert_to_grid(label_set, H_out, W_out)  # (Ntot, Hout, Wout, 6)
        imgs = [img for img, _ in img_set]
        imgs_tensor = torch.stack(imgs, dim=0)
        tensor_data = TensorDataset(imgs_tensor, labels_tensor)
        output_datasets.append(tensor_data)

    return tuple(output_datasets)

train, val, test = load_data(2,3) 
print(f'Train size: {len(train)}')
print(f'Val size: {len(val)}')
print(f'Test size: {len(test)}')  


print(train[0])




### Normalize Images

### Training

In [ ]:
# TODO

### Prediction

In [ ]:
# TODO

### Model selection and evaluation

In [ ]:
# TODO

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def get_map_results(model, eval_loader, device):
    '''
        Helper functions to get predictions and targets in the format required for mAP calculation.
        Depending on your data processing and model architecture this function can either be used as is, 
        modified to fit your needs or used as a blue print for a rewrite.
        Here it is assussmed that the image has been divide into a 2 x 3 grid.
        ----------------------------------------------------------
        Run through the data in the dataloader and collect predicitions and targets for mAP calculation.

        torchmetric mAP expects predictions and targets in the format:
        preds = [
           { "boxes": tensor([[x1, y1, x2, y2], ...]), "scores": tensor([score1, score2, ...]), "labels": tensor([label1, label2, ...])},
            ...   ]
        and targets = [
            { "boxes": tensor([[x1, y1, x2, y2], ...]), "labels": tensor([label1, label2, ...])},
            ...   ]
        where each dict in the list corresponds to one image in the dataset and contains the predicted and true results
    '''
    model.eval()
    with torch.no_grad():
        preds = []
        targets = []
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            outputs = outputs.permute(0, 2, 3, 1)                               # (B, 7, 2, 3) → (B, 2, 3, 7)
            outputs = outputs.reshape(outputs.shape[0], -1, outputs.shape[-1])  # (B, 2, 3, 7) → (B, 6, 7)
            labels = labels.reshape(labels.shape[0], -1, labels.shape[-1])      # (B, 2, 3, 6) → (B, 6, 6)
            for output, label in zip(outputs, labels):
                pred_boxes = []
                pred_scores = []
                pred_labels = []
                target_boxes = []
                target_labels = []
                # collect predicted boxes, scores and labels for the current image
                for i, cell_output in enumerate(output):
                    pred_object_presence = (torch.sigmoid(cell_output[0]) > 0.5) * 1.0
                    if pred_object_presence == 1:
                        # get propability of object presence and class probabilities to compute detection score for mAP calculation
                        obj_prop = torch.sigmoid(cell_output[0]).item()
                        class_prop = F.softmax(cell_output[5:], dim=0)
                        pred_label = torch.argmax(class_prop)
                        detect_score = obj_prop * class_prop[pred_label]
                        # convert from local to global coordinates before we can compare with the labels and compute IoU for mAP calculation
                        bbox_global = local_to_global(i // 3, i % 3, cell_output[1:5])
                        bbox_xyxy = xywh_to_xyxy(bbox_global)
                        bbox_xyxy = torch.stack(bbox_xyxy)
                        # collect predicted boxes, scores and labels for the current image
                        pred_boxes.append(bbox_xyxy)
                        pred_scores.append(detect_score)
                        pred_labels.append(pred_label)
                # collect true boxes and labels for the current image
                for i, cell_label in enumerate(label):
                    true_object_presence = cell_label[0]
                    if true_object_presence == 1:
                        bbox_global = local_to_global(i // 3, i % 3, cell_label[1:5])
                        bbox_xyxy = xywh_to_xyxy(bbox_global)
                        bbox_xyxy = torch.stack(bbox_xyxy)
                        target_boxes.append(bbox_xyxy)
                        target_labels.append(int(cell_label[-1]))
                # store predictions and targets for the current image in the format required for mAP calculation
                # if there are no predicted boxes, we need to create an empty tensor for the boxes, scores and labels to avoid errors in the mAP calculation
                if len(pred_boxes) == 0:
                    pred_dict = {
                        "boxes": torch.zeros((0, 4), device=device),
                        "scores": torch.zeros((0,), device=device),
                        "labels": torch.zeros((0,), dtype=torch.long, device=device),
                    }
                    preds.append(pred_dict)
                else:
                    pred_dict = {
                        "boxes": torch.stack(pred_boxes),
                        "scores": torch.tensor(pred_scores, device=device),
                        "labels": torch.tensor(pred_labels, device=device),
                    }
                    preds.append(pred_dict)
                # if there are no true boxes, we need to create an empty tensor for the boxes and labels to avoid errors in the mAP calculation            
                if len(target_boxes) == 0:
                    target_dict = {
                        "boxes": torch.zeros((0, 4), device=device),
                        "labels": torch.zeros((0,), dtype=torch.long, device=device),
                    }
                    targets.append(target_dict)
                else:
                    target_dict = {
                        "boxes": torch.stack(target_boxes),
                        "labels": torch.tensor(target_labels, device=device),
                    }
                    targets.append(target_dict)
    
    # compute mAP using torchmetrics
    metric = MeanAveragePrecision(iou_type="bbox")
    metric.update(preds, targets)
    results = metric.compute()
    # results is a dict with the mAP results for different IoU thresholds and the overall mAP
    return results        

def local_to_global(i, j, bb, width=60, height=48, cols=3, rows=2):
    x, y, w, h = bb
    # get the dimensions of a single grid cell
    cell_width, cell_height = width / cols, height / rows
    # convert from local to global coordinates
    global_x = x * cell_width + j * cell_width
    global_y = y * cell_height + i * cell_height
    global_w = w * cell_width
    global_h = h * cell_height

    return global_x, global_y, global_w, global_h

def xywh_to_xyxy(bb):
    # convert from center format to box format
    x_center, y_center, w, h = bb
    x1 = x_center - w/2
    y1 = y_center - h/2
    x2 = x_center + w/2
    y2 = y_center + h/2
    return x1, y1, x2, y2   